In [1]:
import pandas as pd
from collections import Counter
import duckdb

# Create CSV with Napoleon Bonapartes Targets

In [2]:
# Get data with old Spotlight predictions
fullmatch_df = pd.read_csv('data/SpotlightWeightSource_0102_0505_fullmatch_corrected.csv')
fullmatch_df.head()

,language_code,source_wikidata_id,target_wikidata_id,weight
0,en,Q6882,Q53003,0.0
1,en,Q6882,Q5912,0.0
2,en,Q6882,Q212719,0.0
3,en,Q6882,Q514998,0.0
4,en,Q6882,Q210134,0.0


In [3]:
# Only keep Napoleon's (Q517) targets
filtered_df = fullmatch_df[(fullmatch_df["source_wikidata_id"]=="Q517") & (fullmatch_df["language_code"]=="en")]
filtered_df.shape

(403, 4)

In [4]:
filtered_df

,language_code,source_wikidata_id,target_wikidata_id,weight
103760,en,Q517,Q708011,0.0
103761,en,Q517,Q8442,0.0
103762,en,Q517,Q152463,0.0
103763,en,Q517,Q157099,0.0
103764,en,Q517,Q208086,0.0
...,...,...,...,...
7415973,en,Q517,Q1685301,0.0
7415974,en,Q517,Q153522,1.0
7415975,en,Q517,Q155002,0.0
7415976,en,Q517,Q37134,1.0


In [5]:
# List of napoleons targets
napo_target_ids = filtered_df["target_wikidata_id"].values.tolist()

In [7]:
entities = pd.read_csv('data/entities_filtered_by_languages.csv')

# Only keep people we want
napo_sources = entities[entities['wikidata_id'].isin(napo_target_ids)]

napo_sources.to_csv('data/napo_targets.csv', index=False)
napo_sources.head()

,wikidata_id,birth,bigperiod_birth,un_subregion,gender
26,Q1003655,1769.0,4.Mid Modern Period 1751-1900AD,Western Europe,Male
51,Q10088,1750.0,3.Early Modern Period 1501-1750AD,South Asia incl. Indian Peninsula,Male
105,Q101638,1759.0,4.Mid Modern Period 1751-1900AD,Western Europe,Female
324,Q103556,973.0,2.Post-Classical History 501-1500AD,Western Europe,Male
797,Q1063401,1750.0,3.Early Modern Period 1501-1750AD,Western Europe,Male


# Napoleon's Ground Truth from Human Signal 

In [8]:
gt_napoleon_targets = pd.read_csv('data/weight_files/ground_truth_raw/GT_napoleon_out_edges_raw.csv')

In [9]:
gt_napoleon_targets = gt_napoleon_targets.drop_duplicates(subset='id', keep='first').reset_index(drop=True)

In [10]:
names = gt_napoleon_targets["entity"].str.extractall(r'"taxonomy":\[\["(.*?)"\]\]')[0]

weights = (
    names.value_counts()
         .rename_axis("Name")
         .reset_index(name="GT_weight")
)
print(weights)

                                   Name  GT_weight
0        Alexander I of Russia (Q15193)         14
1                    Pius VII (Q124754)         13
2                   Napoleon II (Q7723)         12
3    Joséphine de Beauharnais (Q171480)         12
4              Joseph Bonaparte (Q7726)         11
..                                  ...        ...
100           Mikhail Kutuzov (Q185801)          1
101         Feodor Rostopchin (Q742036)          1
102  Claude François de Malet (Q589205)          1
103        Auguste de Marmont (Q312676)          1
104            Marie Walewska (Q237931)          1

[105 rows x 2 columns]


In [11]:
weights[["Name", "ID"]] = weights["Name"].str.extract(r"^(.*) \((Q\d+)\)$")
weights["Name"] = weights["Name"].str.replace(",", "", regex=False)
napoleon_targets_gt = weights[["Name", "ID", "GT_weight"]]
napoleon_targets_gt

,Name,ID,GT_weight
0,Alexander I of Russia,Q15193,14
1,Pius VII,Q124754,13
2,Napoleon II,Q7723,12
3,Joséphine de Beauharnais,Q171480,12
4,Joseph Bonaparte,Q7726,11
...,...,...,...
100,Mikhail Kutuzov,Q185801,1
101,Feodor Rostopchin,Q742036,1
102,Claude François de Malet,Q589205,1
103,Auguste de Marmont,Q312676,1


In [13]:
napoleon_targets_gt.to_csv('data/weight_files/napoleon_out_edges/GT_napoleon_weights.csv', index=False)